# Semantic LLM Cache with LangChain-enVector

This example shows `EnvectorSemanticCache`, a LangChain LLM cache that keeps the
prompts it has seen and their answers in an enVector index. A new prompt that is
close enough in meaning to a stored one is answered from the cache, and nothing
stored is readable by the server.

In this example, we'll see:

- How a paraphrased question is served from the cache without a model request
- How the prompt embeddings, the prompt text and the answers stay encrypted in the index
- How to clear the cache

The model is a plain LLM (`OllamaLLM`, string in, string out), as in LangChain's
own semantic cache examples. A chat model's prompt reaches the cache as a
serialized message list, which embeds very differently; see the README.

## Prerequisites
- enVector server reachable from this notebook environment
- A key path for the key this notebook registers (it is created if missing)
- Ollama running locally, with internet access to pull the two models used below
- `langchain-envector` and `langchain-ollama` packages installed

### Install packages

`langchain-envector` provides the cache; `langchain-ollama` lets LangChain use
the models served by Ollama.

In [ ]:
# !pip install langchain-envector langchain-ollama

## Configure connection and models
Fill in your enVector connection details. The embedding model turns prompts into
vectors; the LLM answers them. Both run locally through Ollama here, so no API
key is needed.

In [ ]:
import os

address = os.getenv("ENVECTOR_ADDRESS", "0.0.0.0:50050")
token = os.getenv("ENVECTOR_ACCESS_TOKEN", "")

key_path = "./keys"
key_id = "semantic_cache_key"
index_name = "llm_cache_demo"
embedding_model = "all-minilm:l6-v2"
llm_model = "gemma3:270m"

print(f"Using key '{key_id}'")
print(f"Using index '{index_name}'")
print(f"Embedding model: {embedding_model}")
print(f"LLM: {llm_model}")

## Pull the Ollama models
Pulling them up front keeps the timings below about the cache rather than about downloads.

In [ ]:
!ollama pull {embedding_model}
!ollama pull {llm_model}

## Set up the embedding model and the LLM
The embedding model's output size becomes the index dimension. The LLM answers
the questions; `temperature=0` keeps its answers stable.

In [ ]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM

embeddings = OllamaEmbeddings(model=embedding_model)
vector_dim = len(embeddings.embed_query("dimension probe"))
print(f"Embedding dimension: {vector_dim}")

llm = OllamaLLM(model=llm_model, temperature=0)

## Create the cache and register it with LangChain

`EnvectorSemanticCache` takes the same configuration as the `Envector` vector
store. Two settings matter here:

- `IndexSettings.metadata_encryption=True` encrypts the prompt text and the stored
  answers. The prompt embeddings are encrypted in any case.
- `distance_threshold` is the largest cosine distance at which a stored prompt
  still counts as a hit: `0` means an identical prompt, and lower is stricter.
  The meaning and the default, `0.2`, are those of LangChain's `RedisSemanticCache`.

`set_llm_cache` makes every LangChain model request go through this cache from
now on.

In [ ]:
from langchain_core.globals import set_llm_cache

from langchain_envector import EnvectorSemanticCache
from langchain_envector.config import ConnectionConfig, EnvectorConfig, IndexSettings, KeyConfig

config = EnvectorConfig(
    connection=ConnectionConfig(address=address, access_token=token) if token else ConnectionConfig(address=address),
    key=KeyConfig(key_path=key_path, key_id=key_id),
    index=IndexSettings(index_name=index_name, dim=vector_dim, metadata_encryption=True),
    create_if_missing=True,
)
cache = EnvectorSemanticCache(config=config, embeddings=embeddings, distance_threshold=0.2)
set_llm_cache(cache)

## Ask a question, then ask it differently

The first question goes to the model and its answer is stored. The second
question is worded differently but means the same, so the cache serves the
stored answer without a model request. The third question is about something
else and goes to the model again.

In [ ]:
import time


def ask(question: str):
    started = time.time()
    answer = llm.invoke(question)
    print(f"[{time.time() - started:5.2f} s] {question}")
    print(f"          {answer.strip()[:120]}")


ask("What is the capital of France?")
ask("Tell me the capital city of France.")
ask("What is the capital of Germany?")

The second call is answered from the cache: it returns in a fraction of the
time and with the same text as the first, while the first and the third took
a model request each. Had the reworded question been farther than
`distance_threshold` from the stored one, it would have gone to the model like
the third one.

## Clear the cache

`cache.clear()` drops every partition this cache created and leaves the rest of
the index alone. After it, the paraphrase from above is a miss again.

In [ ]:
cache.clear()

ask("Tell me the capital city of France.")

### Clean Up

We can delete the created index and the registered key when they are no longer needed.

In [ ]:
cache.vectorstore.client.ev.drop_index(index_name)

In [ ]:
cache.vectorstore.client.ev.delete_key(key_id)